In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

In [ ]:
from pathlib import Path

# Project root
PROJECT_ROOT = Path("..")

# Database folder
DATABASE_DIR = PROJECT_ROOT / "database"

# Create database folder if it doesn't exist
DATABASE_DIR.mkdir(exist_ok=True)

# SQLite database path
DB_PATH = DATABASE_DIR / "retail_analytics.db"

print("Database path:")
print(DB_PATH)

In [ ]:
conn = sqlite3.connect(DB_PATH)

print("SQLite database connected successfully.")

In [ ]:
customer_segments = pd.read_csv(
    PROJECT_ROOT / "outputs" / "ml" / "customer_segments.csv"
)

demand_forecast = pd.read_csv(
    PROJECT_ROOT / "outputs" / "ml" / "demand_forecast.csv"
)

forecast_comparison = pd.read_csv(
    PROJECT_ROOT / "outputs" / "ml" / "forecast_comparison.csv"
)

model_comparison = pd.read_csv(
    PROJECT_ROOT / "outputs" / "ml" / "model_comparison.csv"
)

print("Customer Segments:", customer_segments.shape)
print("Demand Forecast:", demand_forecast.shape)
print("Forecast Comparison:", forecast_comparison.shape)
print("Model Comparison:", model_comparison.shape)

In [ ]:
print("Customer Segments:")
print(customer_segments.columns.tolist())

print("\nDemand Forecast:")
print(demand_forecast.columns.tolist())

print("\nForecast Comparison:")
print(forecast_comparison.columns.tolist())

print("\nModel Comparison:")
print(model_comparison.columns.tolist())

In [ ]:
customer_segments.to_sql(
    "customer_segments",
    conn,
    if_exists="replace",
    index=False
)

demand_forecast.to_sql(
    "demand_forecast",
    conn,
    if_exists="replace",
    index=False
)

forecast_comparison.to_sql(
    "forecast_comparison",
    conn,
    if_exists="replace",
    index=False
)

model_comparison.to_sql(
    "model_comparison",
    conn,
    if_exists="replace",
    index=False
)

print("All ML tables created successfully.")

In [ ]:
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

tables = pd.read_sql_query(query, conn)

tables

In [ ]:
query = """
SELECT 'customer_segments' AS table_name, COUNT(*) AS row_count
FROM customer_segments

UNION ALL

SELECT 'demand_forecast', COUNT(*)
FROM demand_forecast

UNION ALL

SELECT 'forecast_comparison', COUNT(*)
FROM forecast_comparison

UNION ALL

SELECT 'model_comparison', COUNT(*)
FROM model_comparison;
"""

row_counts = pd.read_sql_query(query, conn)

row_counts

In [ ]:
query = """
SELECT *
FROM customer_segments
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)

result

In [ ]:
query = """
SELECT
    customer_segment,
    COUNT(*) AS customers,
    ROUND(AVG(total_spent), 2) AS avg_spending,
    ROUND(AVG(avg_order_value), 2) AS avg_order_value,
    ROUND(AVG(recency_days), 2) AS avg_recency
FROM customer_segments
GROUP BY customer_segment
ORDER BY customers DESC;
"""

segment_summary = pd.read_sql_query(query, conn)

segment_summary

In [ ]:
query = """
SELECT
    customer_unique_id,
    total_orders,
    total_spent,
    avg_order_value,
    recency_days
FROM customer_segments
WHERE customer_segment = 'VIP / Loyal'
ORDER BY total_spent DESC
LIMIT 20;
"""

vip_customers = pd.read_sql_query(query, conn)

vip_customers

In [ ]:
query = """
SELECT
    Model,
    ROUND(MAE, 2) AS MAE,
    ROUND(MSE, 2) AS MSE,
    ROUND(RMSE, 2) AS RMSE,
    ROUND(R2, 3) AS R2
FROM model_comparison
ORDER BY RMSE ASC;
"""

model_results = pd.read_sql_query(query, conn)

model_results

In [ ]:
query = """
SELECT
    Date,
    Actual,
    Predicted,
    Residual
FROM demand_forecast
ORDER BY Date
LIMIT 20;
"""

forecast_data = pd.read_sql_query(query, conn)

forecast_data

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

In [ ]:
processed_dir = PROJECT_ROOT / "data" / "processed"

files = list(processed_dir.glob("*.csv"))

for file in files:
    print(file.name)

In [ ]:
# Load processed retail datasets into SQLite

processed_tables = {
    "customers": "customers_features.csv",
    "orders": "orders_features.csv",
    "order_items": "order_items_features.csv",
    "payments": "payments_features.csv",
    "products": "products_features.csv",
    "reviews": "reviews_features.csv",
    "sellers": "sellers_features.csv",
    "customer_segment_summary": "customer_segment_summary.csv"
}

for table_name, file_name in processed_tables.items():
    
    file_path = processed_dir / file_name
    
    df = pd.read_csv(file_path)
    
    df.to_sql(
        table_name,
        conn,
        if_exists="replace",
        index=False
    )
    
    print(f"{table_name}: {df.shape} → loaded successfully")

print("\nAll processed retail tables loaded successfully.")

In [ ]:
query = """
SELECT
    name AS table_name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

tables = pd.read_sql_query(query, conn)

tables

In [ ]:
# Option A: Inspect schema using SQLite PRAGMA
columns = pd.read_sql_query("PRAGMA table_info(orders);", conn)
print(columns["name"].tolist())

# Option B: Fetch one row to check pandas dataframe columns
print(pd.read_sql_query("SELECT * FROM orders LIMIT 1", conn).columns.tolist())

In [ ]:
query = """
SELECT
    name AS table_name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

tables = pd.read_sql_query(query, conn)

tables

In [ ]:
query = """
SELECT 'customers' AS table_name, COUNT(*) AS row_count
FROM customers

UNION ALL

SELECT 'orders', COUNT(*)
FROM orders

UNION ALL

SELECT 'order_items', COUNT(*)
FROM order_items

UNION ALL

SELECT 'payments', COUNT(*)
FROM payments

UNION ALL

SELECT 'products', COUNT(*)
FROM products

UNION ALL

SELECT 'reviews', COUNT(*)
FROM reviews

UNION ALL

SELECT 'sellers', COUNT(*)
FROM sellers

UNION ALL

SELECT 'customer_segment_summary', COUNT(*)
FROM customer_segment_summary

UNION ALL

SELECT 'customer_segments', COUNT(*)
FROM customer_segments

UNION ALL

SELECT 'demand_forecast', COUNT(*)
FROM demand_forecast

UNION ALL

SELECT 'forecast_comparison', COUNT(*)
FROM forecast_comparison

UNION ALL

SELECT 'model_comparison', COUNT(*)
FROM model_comparison

ORDER BY table_name;
"""

row_counts = pd.read_sql_query(query, conn)

row_counts

In [ ]:
query = """
SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    o.order_status,
    o.order_purchase_timestamp
FROM orders AS o
INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
LIMIT 20;
"""

orders_customers = pd.read_sql_query(query, conn)

orders_customers

In [ ]:
query = """
SELECT
    o.order_id,
    oi.order_item_id,
    oi.product_id,
    p.product_category_name,
    oi.price,
    oi.freight_value
FROM orders AS o
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id
INNER JOIN products AS p
    ON oi.product_id = p.product_id
LIMIT 20;
"""

order_product_data = pd.read_sql_query(query, conn)

order_product_data

In [ ]:
# What this query tells us

# # For each product category:

# # total_orders → number of unique orders containing that category
# # total_items_sold → number of items sold
# # total_sales → sum of product prices
# # avg_item_price → average selling price

# We're using SUM(oi.price) here, so total_sales means product sales value based on item prices. It does not include freight or claim profit.

In [ ]:
query = """
SELECT
    p.product_category_name,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(oi.order_item_id) AS total_items_sold,
    ROUND(SUM(oi.price), 2) AS total_sales,
    ROUND(AVG(oi.price), 2) AS avg_item_price
FROM orders AS o
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id
INNER JOIN products AS p
    ON oi.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY total_sales DESC
LIMIT 20;
"""

category_sales = pd.read_sql_query(query, conn)
category_sales

In [ ]:
query = """
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_spent,
    ROUND(AVG(oi.price), 2) AS avg_item_price
FROM customers AS c
INNER JOIN orders AS o
    ON c.customer_id = o.customer_id
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_unique_id
ORDER BY total_spent DESC
LIMIT 20;
"""

top_customers = pd.read_sql_query(query, conn)
top_customers

In [ ]:
# # What this query does

# # The joins follow:

# # customers
# #      ↓ customer_id
# # orders
# #      ↓ order_id
# # order_items
# #      ↓
# # customer spending

# # So for each unique customer, we calculate:

# # Column	Meaning
# # customer_unique_id	Actual unique customer
# # total_orders	Number of distinct orders
# # total_spent	Total amount spent on items
# # avg_item_price	Average price of purchased items
# # We use:

# COUNT(DISTINCT o.order_id)

# because one order can contain multiple order items. Without DISTINCT, the same order could be counted multiple times.

# And:

# SUM(oi.price)

# gives the total item value purchased by that customer.

# Important for our project

# This SQL analysis is not replacing our ML customer segmentation. It is an additional business-analysis layer.Later, we can connect these insights with the segmentation we already built using customer_features and the 4 customer segments.

In [ ]:
query = """
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_spent,
    ROUND(AVG(oi.price), 2) AS avg_item_price
FROM customers AS c
INNER JOIN orders AS o
    ON c.customer_id = o.customer_id
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_unique_id
ORDER BY total_spent DESC
LIMIT 20;
"""

top_customers = pd.read_sql_query(query, conn)
top_customers

In [ ]:
query = """
SELECT
    s.seller_id,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    COUNT(oi.order_item_id) AS total_items_sold,
    ROUND(SUM(oi.price), 2) AS total_sales,
    ROUND(AVG(oi.price), 2) AS avg_item_price
FROM sellers AS s
INNER JOIN order_items AS oi
    ON s.seller_id = oi.seller_id
GROUP BY s.seller_id
ORDER BY total_sales DESC
LIMIT 20;
"""

top_sellers = pd.read_sql_query(query, conn)
top_sellers

In [ ]:
# Step 17.6 — Product Category Performance

query = """
SELECT
    p.product_category_name,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    COUNT(oi.order_item_id) AS total_items_sold,
    ROUND(SUM(oi.price), 2) AS total_sales,
    ROUND(AVG(oi.price), 2) AS avg_item_price
FROM products AS p
INNER JOIN order_items AS oi
    ON p.product_id = oi.product_id
GROUP BY p.product_category_name
ORDER BY total_sales DESC
LIMIT 20;
"""

top_categories = pd.read_sql_query(query, conn)
top_categories
# This tells us which product categories generate the most revenue.

In [ ]:
query = """
SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS month,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_sales
FROM orders AS o
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id
GROUP BY month
ORDER BY month;
"""

monthly_sales = pd.read_sql_query(query, conn)
monthly_sales

# This gives us the project's time-based sales trend.